In [1]:
# ════════════════════════════════════════════════════════════════════════════════════════════
# MatchScoutIA — TEST DE FAISABILITÉ : BASKET (enfants, petit terrain, filmé au téléphone) — 26/11/2026
# Ne construit rien : mesure ce que le SOCLE actuel sait faire sur cette vidéo, pour décider de la suite.
#   1. RF-DETR (même détecteur que le football) : joueurs et ballon visibles sur combien d'images ?
#   2. Gemini, image par image : phase de jeu, panier visible, score affiché, couleurs des équipes, caméra qui bouge ?
#   3. Gemini, sur des SÉQUENCES de 20 s : « un panier est-il marqué ? » (autour des paniers connus si vous en donnez
#      quelques-uns, et sur des séquences prises au hasard pour mesurer les fausses alertes)
# ENTRÉES : le dataset de la vidéo de basket ; mziparameter (clé Gemini). GPU conseillé (RF-DETR), pas obligatoire.
# COÛT ESTIMÉ : moins de 1 $ de Gemini (≈ 20 questions sur une image + ≈ 10 sur des séquences). Le coût réel s'affiche.
# À GARDER : /kaggle/working/test_basket.zip (images annotées + réponses de Gemini + bilan)
# ════════════════════════════════════════════════════════════════════════════════════════════
import glob, json, os, random, re, subprocess, sys, time, zipfile

# ── RÉGLAGES ───────────────────────────────────────────────────────────────────────────────────
VIDEO = "/kaggle/input/datasets/michaelziant/basketjeune/basketJeune.mp4"   # match de jeunes, règles adultes (31 min)
DEBUT_S, DUREE_S = 0, 1900 # TOUT le match
# (facultatif mais très utile) quelques paniers que VOUS avez vus dans cette portion, en heure de la vidéo :
# 26/11 : vérité donnée par l'utilisateur (heure vidéo → numéro et équipe du marqueur)
# 26/11 : vérité de l'utilisateur pour basketJeune.mp4 (heure vidéo → marqueur). 14:51 : « annulé je pense » → ni
# compté comme manqué, ni comme fausse alerte (INCERTAINS)
MARQUEURS = {"01:04": "31 vert", "02:27": "3 blanc", "07:39": "30 vert", "08:09": "32 vert", "11:59": "5 blanc",
             "12:49": "30 vert", "14:06": "5 blanc", "16:26": "23 vert", "16:41": "3 blanc", "18:44": "11 vert",
             "21:03": "14 blanc", "21:14": "31 vert", "21:58": "32 vert", "24:28": "11 vert", "24:44": "3 blanc",
             "25:32": "11 vert", "26:00": "11 vert", "27:30": "5 blanc", "28:05": "11 vert", "29:42": "3 blanc",
             "30:10": "3 blanc", "30:26": "13 vert"}
PANIERS_CONNUS = list(MARQUEURS)
INCERTAINS = []
# 26/11 : vérifiés par l'utilisateur — 14:51 panier ANNULÉ, 20:49 lancer franc RATÉ (le 2e essai les annonçait)
ANNULES_OU_RATES = ["14:51", "20:49"]
# 26/11 (7e essai) : fenêtre au cercle élargie (−1 → +4 s) ; contrôle de la REPRISE après chaque tir (remise en jeu de
# l'autre équipe = panier ; jeu qui continue = raté ; coup de sifflet / remise sur le côté = faute, panier annulé ?)
FENETRE_CERCLE = (-1.0, 4.0)
CONTROLE_REPRISE = True
# 26/11 : ESSAI CIBLÉ, presque gratuit. L'étape 1 (repérage des tirs) est relue dans le cache du run précédent (0 $ :
# ajouter gemini_cache_basket.jsonl du run précédent en ENTRÉE). Les étapes 2 et reprise ne sont faites QUE pour les
# tirs proches de ces instants : les 7 manqués, les 2 fausses alertes, et 3 paniers trouvés (pour vérifier qu'ils le
# restent). Vide = tous les tirs.
# 26/11 : MODE « DIFFÉRÉ » (Gemini Flex, −50 %, comme l'offre « sous 48 h ») : chaque question peut attendre plusieurs
# minutes chez Google → les questions sont posées EN PARALLÈLE (sinon le test durerait des heures)
FLEX = False   # 26/11 : tests à petit coût → tarif standard, plus rapide
PARALLELE = 32
CIBLES = []                # 26/11 (9e essai) : TOUT le match
# 27/11 : essai ciblé de la mosaïque — tous les vrais paniers + les fausses alertes du 10e essai (la question économique
# et le repérage sont relus du cache : seule la mosaïque est payée)
CIBLES_12E_ESSAI = ["01:04", "02:27", "07:39", "08:09", "11:59", "12:49", "14:06", "16:41", "18:44", "21:03", "21:14", "21:58",
          "24:28", "24:44", "25:32", "26:00", "27:30", "28:05", "29:42", "30:10", "30:26",
          "05:10", "05:58", "16:08", "14:51", "20:49", "01:32", "03:02", "10:50", "19:20"]
CIBLES = []                # 13e essai : tout le match
# 26/11 (9e essai) : VERSION ÉCONOMIQUE ET PRODUIT
#   • pour CHAQUE tir, une question bon marché (2 images avant le tir + 7 après, petites) : type de tir (3 pts / 2 pts /
#     lancer franc), tireur, passeur décisif, reprise du jeu, et rebondeur si le tir est raté ;
#   • la vérification au cercle (chère) SEULEMENT si la reprise n'est pas claire (lancers francs, coup de sifflet,
#     incertain) ; « remise en jeu par l'autre équipe » = panier ; « le jeu continue » = raté (on accepte d'en rater) ;
#   • RÉSUMÉ ≤ 8 min : 3 pts, puis 2 pts, puis lancers francs ; séquences de 8 s ;
#   • MONTAGE D'UN JOUEUR (exemple) : ses paniers, ses passes décisives, ses rebonds.
ECONOMIQUE = True
RESUME_MAX_S = 8 * 60
SEQ_AVANT_S, SEQ_APRES_S = 5, 3
JOUEUR_TEST = ("11", "vert")
# 26/11 (10e essai) : le cercle pour TOUS les tirs (il évite les fausses alertes 01:42, 02:06, 05:10, 17:44 et retrouve
# 26:00, 27:30, 28:05, que la reprise seule ratait), mais en BON MARCHÉ : gros plans réduits à 320 px et 4 images/s.
# La question économique (distance, tireur, passeur, reprise, rebondeur) reste pour TOUS les tirs (relue du cache).
CERCLE_POUR_TOUS = True
CERCLE_TAILLE_MAX = 320
CERCLE_PAS_S = 0.25
# 27/11 (11e essai) : MESURÉ au 10e essai — chaque image coûte ≈ 1 100 jetons QUELLE QUE SOIT SA TAILLE (gros plans de
# 320 px : 23 200 jetons pour 21 images ; question économique : 10 200 pour 9 images) + ≈ 2 200 jetons de « réflexion »
# par réponse. Réduire la taille ne sert donc à rien : il faut réduire le NOMBRE d'images. Variante testée :
#   « mosaique » : les 21 gros plans réunis en UNE image (grille, chaque case marquée de son heure)
CERCLE_VARIANTE = "mosaique"     # "mosaique" ou "" (21 images séparées, comme au 10e essai)
# 27/11 (12e essai) — 11e essai : mosaïque ≈ 1 200 jetons au lieu de 23 200 (19× moins), 9 paniers vus, AUCUNE fausse
# alerte, mais 6 paniers vus en images séparées lui échappent (cases trop petites). Donc :
#   • DEUX mosaïques de 12 cases plus grandes (≈ 2 400 jetons) au lieu d'une de 21 ;
#   • RATTRAPAGE : si les mosaïques disent « non » mais que la reprise montre une remise en jeu de l'autre équipe,
#     on repose la question avec les 21 gros plans séparés (cher, mais seulement pour ces quelques tirs).
MOSAIQUE_CASES = 12
MOSAIQUE_COTE = 420
RATTRAPAGE_IMAGES_SEPAREES = True
# 27/11 (13e essai) : la mosaïque aussi pour le REPÉRAGE des tirs (11 images → 1) et la QUESTION ÉCONOMIQUE (9 → 1),
# réflexion réduite ; TOUT le match, pour le bilan et le coût définitifs
MOSAIQUE_REPERAGE = True
MOSAIQUE_ECO = True
# 27/11 (14e essai) — 13e essai : 10/22 seulement. La mosaïque situe MAL le panier (gros plans à côté) et juge mal la
# reprise. Donc REPÉRAGE EN DEUX TEMPS : la mosaïque (2 × 6 cases plus grandes) ne donne QUE le moment du tir ; le panier
# visé est ensuite situé sur UNE image en pleine taille au moment du tir. Question économique : images SÉPARÉES (comme
# au 12e essai — en bonne partie relue du cache quand le tir tombe au même instant).
MOSAIQUE_REPERAGE = "temps"
MOSAIQUE_ECO = False
REFLEXION_BASSE = True           # réflexion réduite pour ces questions simples (oui / non)
CONTROLE_REPRISE = not ECONOMIQUE   # en version économique, la reprise est déjà dans la question bon marché
N_SEQUENCES_HASARD = 5     # séquences sans panier connu, pour mesurer les fausses alertes
# 26/11 : BALAYAGE de toute la portion par séquences de 20 s qui se chevauchent (une toutes les 10 s) : c'est ce que
# ferait un vrai moteur basket ; mesure les paniers trouvés, les fausses alertes et le numéro du marqueur
BALAYAGE = True
SAUTER_ETAPES_1_2 = True   # déjà faites au 1er essai
# 26/11 (3e essai) : envoyer à Gemini un VRAI EXTRAIT VIDÉO de 20 s (FPS_VIDEO images/s) au lieu de 11 images espacées de
# 2 s : le ballon ne traverse le filet que pendant une fraction de seconde (00:10 et 03:17 manqués au 2e essai)
SEQUENCE_EN_VIDEO = False    # 3e essai (extraits vidéo) : 1/4 seulement — Gemini les a lus en trop peu de détails
FPS_VIDEO = 4
# 26/11 (4e essai) : images comme au 2e essai (2/4, 0 fausse alerte) mais UNE PAR SECONDE (au lieu d'une toutes les
# 2 s) et la RÈGLE DE LA REPRISE (après un panier, remise en jeu par l'autre équipe) dans la question
PAS_IMAGES_S = 1
# 26/11 (5e essai) : DEUX ÉTAPES. Les paniers de 00:10 et 03:17 sont BIEN visibles dans le filet (captures de
# l'utilisateur), mais le cercle ne fait que ~30 px sur une image réduite à 960 px → « rebondit sur le cercle ».
#   étape 1 : balayage toutes les 2 s (comme le 2e essai) → Gemini liste les TIRS (instant + position du panier visé) ;
#   étape 2 : pour chaque tir, images en PLEINE RÉSOLUTION, recadrées autour du panier, 5 par seconde de −1 s à +2 s
#             → « le ballon passe-t-il dans le filet ? » (ne dépend plus de la reprise du jeu)
DEUX_ETAPES = True
# ───────────────────────────────────────────────────────────────────────────────────────────────

REPO = "/kaggle/working/ProjectAIMika"
SORTIE = "/kaggle/working/test_basket"
os.makedirs(SORTIE, exist_ok=True)
if not os.path.isdir(REPO):
    os.system(f"git clone -q --depth 1 https://github.com/mikarocco-droid/ProjectAIMika.git {REPO}")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "rfdetr", "google-genai", "opencv-python-headless"], check=False)
os.chdir(REPO); sys.path.insert(0, REPO)
for l in open(next(iter(glob.glob("/kaggle/input/**/param.env", recursive=True))), encoding="utf-8"):
    if l.strip().startswith("GEMINI_API_KEY="):
        os.environ["GEMINI_API_KEY"] = l.split("=", 1)[1].strip().strip('"').strip("'")
os.environ["SCOUTIA_GEMINI_CACHE_OUT"] = f"{SORTIE}/gemini_cache_basket.jsonl"
if FLEX:
    os.environ["SCOUTIA_GEMINI_TIER"] = "flex"
import ai.gemini_validator as gv          # VERROU d'abord : modèle unique gemini-3.8-flash + cache + comptage des coûts
import cv2
import numpy as np

if not VIDEO:
    cands = [p for p in glob.glob("/kaggle/input/**/*.mp4", recursive=True) + glob.glob("/kaggle/input/**/*.MP4", recursive=True)
             + glob.glob("/kaggle/input/**/*.mov", recursive=True) + glob.glob("/kaggle/input/**/*.MOV", recursive=True)
             if "basket" in p.lower()]
    if not cands:
        raise SystemExit("❌ aucune vidéo dont le chemin contient « basket » : indiquez VIDEO = \"/kaggle/input/…\"")
    VIDEO = cands[0]
cap = cv2.VideoCapture(VIDEO)
fps = cap.get(cv2.CAP_PROP_FPS) or 25.0
duree = cap.get(cv2.CAP_PROP_FRAME_COUNT) / fps
w, h = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fin = min(duree, DEBUT_S + DUREE_S)
print(f"VIDÉO : {VIDEO}\n  {w}x{h} | {fps:.1f} images/s | {duree / 60:.1f} min | portion testée {DEBUT_S}→{fin:.0f} s")
mmss = lambda t: f"{int(t) // 60:02d}:{int(t) % 60:02d}"
sec = lambda h_: int(h_.split(":")[0]) * 60 + int(h_.split(":")[1])


def image_a(t):
    cap.set(cv2.CAP_PROP_POS_MSEC, t * 1000)
    ok, img = cap.read()
    return img if ok else None


# ── 1. RF-DETR : joueurs et ballon ─────────────────────────────────────────────────────────────
print("\n══ 1. DÉTECTION (RF-DETR, comme au football) ══")
if SAUTER_ETAPES_1_2:
    print("  (sautée : déjà faite)")
    instants = []
if not SAUTER_ETAPES_1_2:
    from rfdetr import RFDETRMedium
    modele = RFDETRMedium()
    try:
        modele.optimize_for_inference()
    except Exception:
        pass
    instants = [DEBUT_S + k * 10 for k in range(int((fin - DEBUT_S) // 10))]       # une image toutes les 10 s
n_pers, ballon, tailles = [], 0, []
for i, t in enumerate(instants):
    img = image_a(t)
    if img is None:
        continue
    dets = modele.predict(cv2.cvtColor(img, cv2.COLOR_BGR2RGB), threshold=0.3)
    pers = [b for b, k in zip(dets.xyxy, dets.class_id) if int(k) == 1]       # 1 = personne (COCO, calibré au football)
    bal = [b for b, k in zip(dets.xyxy, dets.class_id) if int(k) == 37]       # 37 = ballon de sport
    n_pers.append(len(pers)); ballon += bool(bal)
    tailles += [float(b[3] - b[1]) / h for b in pers]
    if i % 6 == 0:                                                            # quelques images annotées à regarder
        for b in pers:
            cv2.rectangle(img, (int(b[0]), int(b[1])), (int(b[2]), int(b[3])), (0, 255, 0), 2)
        for b in bal:
            cv2.rectangle(img, (int(b[0]), int(b[1])), (int(b[2]), int(b[3])), (0, 0, 255), 3)
        cv2.imwrite(f"{SORTIE}/detection_{mmss(t).replace(':', 'm')}.jpg", img)
if n_pers:
    print(f"  {len(n_pers)} images | personnes par image : médiane {int(np.median(n_pers))} (min {min(n_pers)}, max {max(n_pers)})"
          f" | BALLON vu sur {100 * ballon / len(n_pers):.0f} % des images"
          f" | taille d'un joueur : {100 * np.median(tailles):.0f} % de la hauteur de l'image (football Veo ≈ 5 à 10 %)")

# ── 2. Gemini sur une image ───────────────────────────────────────────────────────────────────
print("\n══ 2. GEMINI, IMAGE PAR IMAGE ══")
from google import genai
from google.genai import types
client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])
QUESTION_IMAGE = (
    "Youth basketball game filmed with a phone. Answer ONLY with JSON: {\"phase\": \"jeu|lancer franc|remise en jeu|"
    "arret|temps mort|hors match\", \"panier_visible\": true/false, \"les_deux_paniers_visibles\": true/false, "
    "\"score_affiche\": \"texte du tableau de score lisible ou ''\", \"equipe_A\": \"couleur maillot\", "
    "\"equipe_B\": \"couleur maillot\", \"numeros_lisibles\": nombre de numéros de maillot lisibles, "
    "\"cadrage\": \"large|serre|suit l'action\"}")
reponses_img = []
for t in instants[::3][:20]:
    img = image_a(t)
    if img is None:
        continue
    _, jpg = cv2.imencode(".jpg", img, [cv2.IMWRITE_JPEG_QUALITY, 85])
    try:
        r = client.models.generate_content(model="gemini-3.8-flash", contents=[
            QUESTION_IMAGE, types.Part.from_bytes(data=jpg.tobytes(), mime_type="image/jpeg")], config={"temperature": 0})
        txt = r.text or ""
        d = json.loads(txt[txt.index("{"):txt.rindex("}") + 1])
    except Exception as e:
        d = {"erreur": str(e)[:120]}
    d["t"] = mmss(t)
    reponses_img.append(d)
    print(f"  {mmss(t)} : {d}")

# ── 3. Gemini sur des séquences de 20 s : « un panier est-il marqué ? » ───────────────────────
print("\n══ 3. GEMINI, SÉQUENCES DE 20 s : PANIER MARQUÉ ? ══")
QUESTION_SEQ = (
    "These are consecutive frames (timestamps given) of a youth basketball game filmed with a phone (mini-basket, "
    "low hoops). Is a BASKET SCORED (ball going through the hoop) in this sequence? Decisive rule: AFTER A BASKET, play "
    "stops and the OTHER team restarts with a throw-in (inbound pass) from outside the court. If play simply CONTINUES "
    "after the shot (rebound), there was NO basket. If the referee whistles and play restarts from the SIDE line, it is "
    "a foul or the ball went out, NOT a basket. Other clues: the ball drops through the net, the scorers celebrate. "
    "Answer ONLY with JSON: {\"panier\": true/false, \"reprise\": \"remise en jeu par l'autre equipe|le jeu continue|"
    "coup de sifflet, remise sur le cote|pas de tir|incertain\", "
    "\"instant\": \"mm:ss or ''\", \"equipe\": \"maillot couleur or ''\", \"marqueur\": \"numéro du tireur ou ''\", "
    "\"passeur\": \"numéro du passeur décisif ou ''\", \"certitude\": \"haute|moyenne|faible\", "
    "\"preuve\": \"one short sentence\"}")
fenetres = [("connu", sec(p)) for p in PANIERS_CONNUS]
random.seed(1)
hasard = [t for t in range(DEBUT_S + 20, int(fin) - 20, 20) if all(abs(t - sec(p)) > 30 for p in PANIERS_CONNUS)]
fenetres += [("hasard", t) for t in random.sample(hasard, min(N_SEQUENCES_HASARD, len(hasard)))]
reponses_seq = []
if BALAYAGE:
    fenetres = [("balayage", t) for t in range(DEBUT_S + 10, int(fin) - 9, 10)]
QUESTION_TIRS = (
    "These are consecutive frames (timestamps given, every 2 seconds) of a youth basketball game filmed with a phone "
    "(mini-basket, low mobile hoops). List EVERY SHOT at a hoop (any attempt, made or missed). For each: the timestamp of "
    "the frame closest to the release or the ball at the rim, and the bounding box of the TARGETED HOOP (rim + net) on "
    "that frame as [ymin, xmin, ymax, xmax] scaled 0-1000. Answer ONLY with JSON: {\"tirs\": [{\"instant\": \"mm:ss\", "
    "\"panier_box\": [ymin, xmin, ymax, xmax], \"tireur\": \"numéro et couleur ou ''\"}]}")
QUESTION_CERCLE = (
    "These are close-up crops (full resolution, 5 per second, timestamps given) around a basketball hoop, during a shot "
    "in a youth game. Does the BALL GO THROUGH THE NET (basket scored)? A ball touching the rim and falling outside is NOT "
    "a basket. Answer ONLY with JSON: {\"panier\": true/false, \"instant\": \"mm:ss\", \"certitude\": "
    "\"haute|moyenne|faible\", \"preuve\": \"one short sentence\"}")


QUESTION_MOSAIQUE = (
    "These images are GRIDS of close-up crops around a basketball hoop during a shot (youth game), in TIME ORDER: "
    "left to right, then top to bottom, image after image; each crop is labelled with its timestamp (mm:ss.cc), 4 crops "
    "per second. Does "
    "the BALL GO THROUGH THE NET (basket scored)? A ball touching the rim and falling outside is NOT a basket. Answer "
    "ONLY with JSON: {\"panier\": true/false, \"instant\": \"mm:ss\", \"certitude\": \"haute|moyenne|faible\", "
    "\"preuve\": \"one short sentence\"}")


def _mosaique_cases(cases, colonnes, cote):
    """Comme _mosaique, et renvoie aussi la position de chaque case : [(etiq, y0, x0, hauteur, largeur)]."""
    lignes = (len(cases) + colonnes - 1) // colonnes
    haut = max(int(img.shape[0] * cote / max(img.shape[:2])) for _, img in cases)   # pas de bandes blanches inutiles
    mos = np.full((lignes * (haut + 22), colonnes * cote, 3), 255, np.uint8)
    pos = []
    for i, (etiq, img) in enumerate(cases):
        f = cote / max(img.shape[:2])
        img = cv2.resize(img, (max(1, int(img.shape[1] * f)), max(1, int(img.shape[0] * f))))
        r, c = divmod(i, colonnes)
        y, x_ = r * (haut + 22), c * cote
        mos[y + 22:y + 22 + img.shape[0], x_:x_ + img.shape[1]] = img
        cv2.putText(mos, etiq, (x_ + 4, y + 17), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (0, 0, 0), 2)
        pos.append((etiq, y + 22, x_, img.shape[0], img.shape[1]))
    return mos, pos


QUESTION_TIRS_TEMPS = (
    "These images are GRIDS of frames of a youth basketball game (adult rules) filmed with a phone, in TIME ORDER (left "
    "to right, then top to bottom, image after image), one frame every 2 seconds, each labelled with its timestamp "
    "(mm:ss). List EVERY SHOT at a hoop (made or missed, free throws included): the timestamp of the frame closest to "
    "the release or to the ball at the rim, and the shooter if readable. Answer ONLY with JSON: {\"tirs\": "
    "[{\"instant\": \"mm:ss\", \"tireur\": \"numéro et couleur ou ''\"}]}")
QUESTION_PANIER_BOX = (
    "Basketball game frame at {T}, during or just after a shot. Give the bounding box of the hoop (rim + net) the ball "
    "is going to (if only one hoop is visible, that one), as [ymin, xmin, ymax, xmax] scaled 0-1000. Answer ONLY with "
    "JSON: {\"panier_box\": [ymin, xmin, ymax, xmax]}")
QUESTION_TIRS_MOSAIQUE = (
    "This image is a GRID of frames of a youth basketball game (adult rules) filmed with a phone, in TIME ORDER (left to "
    "right, then top to bottom), one frame every 2 seconds, each labelled with its timestamp (mm:ss). List EVERY SHOT at "
    "a hoop (made or missed). For each: the timestamp of the frame closest to the release or the ball at the rim, and "
    "the bounding box of the TARGETED HOOP (rim + net) IN THAT FRAME, as [ymin, xmin, ymax, xmax] scaled 0-1000 on the "
    "WHOLE grid image. Answer ONLY with JSON: {\"tirs\": [{\"instant\": \"mm:ss\", \"panier_box\": [ymin, xmin, ymax, "
    "xmax], \"tireur\": \"numéro et couleur ou ''\"}]}")


def _mosaique(cases, colonnes=6, cote=300):
    """Gros plans réunis en UNE image (grille), chacun marqué de son heure, dans l'ordre du temps."""
    lignes = (len(cases) + colonnes - 1) // colonnes
    mos = np.full((lignes * (cote + 22), colonnes * cote, 3), 255, np.uint8)
    for i, (etiq, img) in enumerate(cases):
        f = cote / max(img.shape[:2])
        img = cv2.resize(img, (max(1, int(img.shape[1] * f)), max(1, int(img.shape[0] * f))))
        r, c = divmod(i, colonnes)
        y, x_ = r * (cote + 22), c * cote
        mos[y + 22:y + 22 + img.shape[0], x_:x_ + img.shape[1]] = img
        cv2.putText(mos, etiq, (x_ + 4, y + 17), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (0, 0, 0), 2)
    return mos


QUESTION_REPRISE = (
    "A shot at a hoop happened at {T} in a basketball game (youth players, adult rules). These frames come AFTER the shot "
    "(timestamps given). How does play continue? \"remise en jeu par l'autre equipe\" = the defending team restarts with "
    "a throw-in from behind the END line under that hoop (this happens after a made basket); \"le jeu continue\" = "
    "rebound and play goes on (missed shot); \"coup de sifflet, remise sur le cote\" = the referee stops play (foul, "
    "violation, out of bounds) and play restarts from the side or with free throws; \"lancers francs\" = free throws "
    "follow. Answer ONLY with JSON: {\"reprise\": \"remise en jeu par l'autre equipe|le jeu continue|coup de sifflet, "
    "remise sur le cote|lancers francs|incertain\", \"preuve\": \"one short sentence\"}")


QUESTION_TIR_ECO = (
    "Basketball game (youth players, adult rules), phone video. A SHOT at a hoop happened around {T}. The first two "
    "frames are just BEFORE the shot, the others come AFTER (timestamps given). Answer ONLY with JSON: "
    "{\"lancer_franc\": true/false, \"distance_m\": estimated distance in metres between the shooter and the hoop at "
    "release (number, e.g. 1.5 for a lay-up, 7 for a long shot), \"tireur\": \"numéro et couleur du maillot ou ''\", "
    "\"passeur\": \"numéro et couleur du passeur décisif ou ''\", "
    "\"reprise\": \"remise en jeu par l'autre equipe|le jeu continue|coup de sifflet, remise sur le cote|lancers francs|"
    "incertain\" (after a MADE basket the defending team throws the ball in from behind the END line; after a miss, "
    "play continues with a rebound), \"rebondeur\": \"numéro et couleur du joueur qui prend le rebond si le tir est "
    "raté, sinon ''\"}")


def demander(contenu, simple=False):
    config = {"temperature": 0}
    if simple and REFLEXION_BASSE:
        config["thinking_config"] = {"thinking_level": "low"}
    try:
        r = client.models.generate_content(model="gemini-3.8-flash", contents=contenu, config=config)
    except Exception as e:
        if "thinking" not in str(e).lower():
            raise
        r = client.models.generate_content(model="gemini-3.8-flash", contents=contenu, config={"temperature": 0})
    txt = r.text or ""
    return json.loads(txt[txt.index("{"):txt.rindex("}") + 1])


if DEUX_ETAPES:
    print("  étape 1 : repérage des tirs (images toutes les 2 s)")
    tirs = []
    from concurrent.futures import ThreadPoolExecutor
    import threading as _th
    _verrou_video = _th.Lock()                     # une seule lecture vidéo à la fois (OpenCV n'aime pas le parallèle)

    def _sequence_tirs(tc):
        if MOSAIQUE_REPERAGE == "temps":
            cases = []
            for t in range(tc - 10, tc + 11, 2):
                with _verrou_video:
                    img = image_a(t)
                if img is not None:
                    cases.append((mmss(t), img))
            if not cases:
                return []
            contenu = [QUESTION_TIRS_TEMPS]
            for i0 in range(0, len(cases), 6):
                mos, _p = _mosaique_cases(cases[i0:i0 + 6], colonnes=3, cote=640)
                _, jpg = cv2.imencode(".jpg", mos, [cv2.IMWRITE_JPEG_QUALITY, 85])
                contenu.append(types.Part.from_bytes(data=jpg.tobytes(), mime_type="image/jpeg"))
            try:
                brut = demander(contenu, simple=True).get("tirs") or []
            except Exception as e:
                print(f"    séquence {mmss(tc)} : {e}")
                return []
            return [{"instant": x.get("instant"), "panier_box": None, "tireur": x.get("tireur", "")} for x in brut]
        if MOSAIQUE_REPERAGE:
            cases = []
            for t in range(tc - 10, tc + 11, 2):
                with _verrou_video:
                    img = image_a(t)
                if img is not None:
                    cases.append((mmss(t), img))
            if not cases:
                return []
            mos, pos = _mosaique_cases(cases, colonnes=4, cote=480)
            Hm, Wm = mos.shape[:2]
            _, jpg = cv2.imencode(".jpg", mos, [cv2.IMWRITE_JPEG_QUALITY, 85])
            try:
                brut = demander([QUESTION_TIRS_MOSAIQUE, types.Part.from_bytes(data=jpg.tobytes(), mime_type="image/jpeg")],
                                simple=True).get("tirs") or []
            except Exception as e:
                print(f"    séquence {mmss(tc)} : {e}")
                return []
            sortie = []
            for x in brut:                 # cadre du panier : coordonnées de la mosaïque → coordonnées de SA case
                b = x.get("panier_box") or []
                if len(b) != 4:
                    continue
                cy, cx = (float(b[0]) + float(b[2])) / 2 * Hm / 1000, (float(b[1]) + float(b[3])) / 2 * Wm / 1000
                case = next((p for p in pos if p[1] <= cy <= p[1] + p[3] and p[2] <= cx <= p[2] + p[4]), None)
                if not case:
                    continue
                etiq, y0, x0, hc, wc = case
                conv = [max(0, min(1000, (float(b[0]) * Hm / 1000 - y0) / hc * 1000)),
                        max(0, min(1000, (float(b[1]) * Wm / 1000 - x0) / wc * 1000)),
                        max(0, min(1000, (float(b[2]) * Hm / 1000 - y0) / hc * 1000)),
                        max(0, min(1000, (float(b[3]) * Wm / 1000 - x0) / wc * 1000))]
                sortie.append({"instant": x.get("instant") or etiq, "panier_box": conv, "tireur": x.get("tireur", "")})
            return sortie
        contenu = [QUESTION_TIRS]
        for t in range(tc - 10, tc + 11, 2):
            with _verrou_video:
                img = image_a(t)
            if img is None:
                continue
            img = cv2.resize(img, (960, int(960 * img.shape[0] / img.shape[1])))
            _, jpg = cv2.imencode(".jpg", img, [cv2.IMWRITE_JPEG_QUALITY, 80])
            contenu += [f"[{mmss(t)}]", types.Part.from_bytes(data=jpg.tobytes(), mime_type="image/jpeg")]
        try:
            return demander(contenu).get("tirs") or []
        except Exception as e:
            print(f"    séquence {mmss(tc)} : {e}")
            return []

    if FLEX:
        _sequence_tirs = gv.vague_flex(_sequence_tirs)       # sinon le verrou garde le tarif standard
    with ThreadPoolExecutor(max_workers=PARALLELE if FLEX else 4) as _pool:
        _listes = list(_pool.map(_sequence_tirs, list(range(DEBUT_S + 10, int(fin) - 9, 20))))
    for _l in _listes:
        for x in _l:
            if ":" in str(x.get("instant", "")) and (len(x.get("panier_box") or []) == 4 or MOSAIQUE_REPERAGE == "temps"):
                try:
                    t_ = sec(x["instant"])
                except Exception:
                    continue
                if all(abs(t_ - y["t"]) > 2 for y in tirs):
                    tirs.append({"t": t_, "box": x.get("panier_box"), "tireur": x.get("tireur", "")})
    if MOSAIQUE_REPERAGE == "temps":
        # 2e temps : le panier visé, situé sur UNE image en pleine taille au moment du tir
        def _localiser(x):
            with _verrou_video:
                img = image_a(x["t"])
            if img is None:
                return None
            _, jpg = cv2.imencode(".jpg", img, [cv2.IMWRITE_JPEG_QUALITY, 88])
            try:
                b = demander([QUESTION_PANIER_BOX.replace("{T}", mmss(x["t"])),
                              types.Part.from_bytes(data=jpg.tobytes(), mime_type="image/jpeg")], simple=True).get("panier_box")
                return b if b and len(b) == 4 else None
            except Exception:
                return None
        if FLEX:
            _localiser = gv.vague_flex(_localiser)
        with ThreadPoolExecutor(max_workers=PARALLELE if FLEX else 4) as _pool:
            _boites = list(_pool.map(_localiser, tirs))
        for x, b in zip(tirs, _boites):
            x["box"] = b
        n_sans = sum(1 for x in tirs if not x["box"])
        tirs = [x for x in tirs if x["box"]]
        print(f"  panier situé sur une image en pleine taille : {len(tirs)} tir(s) ({n_sans} sans panier visible, écartés)")
    print(f"  {len(tirs)} tir(s) repéré(s) : {[mmss(x['t']) + ' ' + str(x['tireur']) for x in sorted(tirs, key=lambda y: y['t'])]}")
    print("  étape 2 : vérification au cercle (pleine résolution, recadrée)")
    if CIBLES:
        tirs = [x for x in tirs if any(abs(x["t"] - sec(c)) <= 6 for c in CIBLES)]
        print(f"  ESSAI CIBLÉ : {len(tirs)} tir(s) vérifié(s) : {[mmss(x['t']) for x in sorted(tirs, key=lambda y: y['t'])]}")
    if ECONOMIQUE:
        def _image_verrou_eco(t):
            with _verrou_video:
                return image_a(t)

        def _tir_eco(x):
            if MOSAIQUE_ECO:
                cases = []
                for t in [x["t"] - 2, x["t"] - 1] + list(range(int(x["t"]) + 1, int(x["t"]) + 14, 2)):
                    img = _image_verrou_eco(t)
                    if img is not None:
                        cases.append((mmss(t), img))
                if not cases:
                    return {"reprise": "incertain"}
                mos, _pos = _mosaique_cases(cases, colonnes=3, cote=640)
                _, jpg = cv2.imencode(".jpg", mos, [cv2.IMWRITE_JPEG_QUALITY, 85])
                q = QUESTION_TIR_ECO.replace("{T}", mmss(x["t"])) + (
                    " The frames are given as ONE GRID image, in time order (left to right, then top to bottom), each "
                    "labelled with its timestamp.")
                try:
                    return demander([q, types.Part.from_bytes(data=jpg.tobytes(), mime_type="image/jpeg")], simple=True)
                except Exception as e:
                    return {"reprise": "incertain", "erreur": str(e)[:80]}
            contenu = [QUESTION_TIR_ECO.replace("{T}", mmss(x["t"]))]
            for t in [x["t"] - 2, x["t"] - 1] + list(range(int(x["t"]) + 1, int(x["t"]) + 14, 2)):
                img = _image_verrou_eco(t)
                if img is None:
                    continue
                larg = 960 if t < x["t"] else 640          # avant le tir : assez grand pour situer le tireur
                img = cv2.resize(img, (larg, int(larg * img.shape[0] / img.shape[1])))
                _, jpg = cv2.imencode(".jpg", img, [cv2.IMWRITE_JPEG_QUALITY, 80])
                contenu += [f"[{mmss(t)}]", types.Part.from_bytes(data=jpg.tobytes(), mime_type="image/jpeg")]
            try:
                return demander(contenu)
            except Exception as e:
                return {"reprise": "incertain", "erreur": str(e)[:80]}

        if FLEX:
            _tir_eco = gv.vague_flex(_tir_eco)
        with ThreadPoolExecutor(max_workers=PARALLELE if FLEX else 4) as _pool:
            _tt = sorted(tirs, key=lambda y: y["t"])
            _eco = list(_pool.map(_tir_eco, _tt))
        douteux = []
        for x, e in zip(_tt, _eco):
            try:
                _dist = float(e.get("distance_m"))
            except (TypeError, ValueError):
                _dist = 3.0
            x.update({"type": "lancer franc" if e.get("lancer_franc") else f"{_dist:.1f} m", "distance": _dist,
                      "lancer_franc": bool(e.get("lancer_franc")), "tireur2": e.get("tireur") or x.get("tireur", ""),
                      "passeur": e.get("passeur", ""), "reprise": e.get("reprise", "incertain"),
                      "rebondeur": e.get("rebondeur", "")})
            r = str(x["reprise"])
            if "remise en jeu" in r:
                x["decision"] = "panier"
            elif "continue" in r:
                x["decision"] = "raté"
            else:
                x["decision"] = "?"
                douteux.append(x)
        print(f"  {len(_tt)} tir(s) décrits (question bon marché) | {len(douteux)} douteux → vérification au cercle")
        tirs = list(_tt) if CERCLE_POUR_TOUS else douteux    # le cercle (plus bas) : tous les tirs, ou les douteux

    def _image_verrou(t):
        with _verrou_video:
            return image_a(t)

    def _verifier_tir(x):
        y0, x0, y1, x1 = [float(v) / 1000 for v in x["box"]]
        cy, cx = (y0 + y1) / 2 * h, (x0 + x1) / 2 * w
        demi = max(140, 2.2 * max((y1 - y0) * h, (x1 - x0) * w))          # cadre ≈ 4 à 5 fois le panier
        contenu = [QUESTION_CERCLE]
        _cases = []
        _pas = CERCLE_PAS_S if CERCLE_POUR_TOUS else 0.2
        for k in range(int(FENETRE_CERCLE[0] / _pas), int(FENETRE_CERCLE[1] / _pas) + 1):
            t = x["t"] + k * _pas
            img = _image_verrou(t)
            if img is None:
                continue
            ya, yb = int(max(0, cy - demi)), int(min(h, cy + demi * 1.6))   # plus de place sous le cercle (filet)
            xa, xb = int(max(0, cx - demi)), int(min(w, cx + demi))
            crop = img[ya:yb, xa:xb]
            if crop.size == 0:
                continue
            if CERCLE_POUR_TOUS and max(crop.shape[:2]) > CERCLE_TAILLE_MAX:      # gros plan réduit (coût)
                _f = CERCLE_TAILLE_MAX / max(crop.shape[:2])
                crop = cv2.resize(crop, (int(crop.shape[1] * _f), int(crop.shape[0] * _f)))
            _etiq = f"{mmss(t)}.{int(round((t % 1) * 100)):02d}"
            if CERCLE_VARIANTE == "mosaique":
                _cases.append((_etiq, crop))
                continue
            _, jpg = cv2.imencode(".jpg", crop, [cv2.IMWRITE_JPEG_QUALITY, 90])
            contenu += [f"[{_etiq}]", types.Part.from_bytes(data=jpg.tobytes(), mime_type="image/jpeg")]
            if k == 0:
                cv2.imwrite(f"{SORTIE}/cercle_{mmss(x['t']).replace(':', 'm')}.jpg", crop)
        _separees = []
        if CERCLE_VARIANTE == "mosaique" and _cases:
            contenu = [QUESTION_MOSAIQUE]
            for i0 in range(0, len(_cases), MOSAIQUE_CASES):
                mos = _mosaique(_cases[i0:i0 + MOSAIQUE_CASES], colonnes=4, cote=MOSAIQUE_COTE)
                cv2.imwrite(f"{SORTIE}/mosaique_{mmss(x['t']).replace(':', 'm')}_{i0 // MOSAIQUE_CASES + 1}.jpg", mos)
                _, jpg = cv2.imencode(".jpg", mos, [cv2.IMWRITE_JPEG_QUALITY, 90])
                contenu.append(types.Part.from_bytes(data=jpg.tobytes(), mime_type="image/jpeg"))
            for etiq, crop in _cases:
                _, jpg = cv2.imencode(".jpg", crop, [cv2.IMWRITE_JPEG_QUALITY, 90])
                _separees += [f"[{etiq}]", types.Part.from_bytes(data=jpg.tobytes(), mime_type="image/jpeg")]
        try:
            d = demander(contenu, simple=True)
            if (CERCLE_VARIANTE == "mosaique" and RATTRAPAGE_IMAGES_SEPAREES and not d.get("panier")
                    and "remise en jeu" in str(x.get("reprise", "")) and _separees):
                d2 = demander([QUESTION_CERCLE] + _separees, simple=True)
                d2["preuve"] = "[rattrapage, images séparées] " + str(d2.get("preuve", ""))
                d = d2 if d2.get("panier") else d
        except Exception as e:
            d = {"erreur": str(e)[:120]}
        d.update({"instant": d.get("instant") or mmss(x["t"]), "marqueur": x["tireur"], "nature": "balayage",
                  "centre": mmss(x["t"])})
        if ":" in str(d.get("instant", "")):
            d["instant"] = str(d["instant"]).split(".")[0]
        if CONTROLE_REPRISE:
            contenu_r = [QUESTION_REPRISE.replace("{T}", mmss(x["t"]))]
            for t in range(int(x["t"]) + 1, int(x["t"]) + 14, 2):
                img = _image_verrou(t)
                if img is None:
                    continue
                img = cv2.resize(img, (640, int(640 * img.shape[0] / img.shape[1])))
                _, jpg = cv2.imencode(".jpg", img, [cv2.IMWRITE_JPEG_QUALITY, 75])
                contenu_r += [f"[{mmss(t)}]", types.Part.from_bytes(data=jpg.tobytes(), mime_type="image/jpeg")]
            try:
                rp = demander(contenu_r)
            except Exception as e:
                rp = {"reprise": "incertain", "preuve": str(e)[:80]}
            d["reprise"] = rp.get("reprise", "incertain")
            # DÉCISION : le cercle d'abord ; la reprise confirme, met en doute (faute → annulé ?) ou rattrape
            if d.get("panier") and "sifflet" in str(d["reprise"]):
                d["statut"] = "à vérifier (annulé ? coup de sifflet après le panier)"
            elif d.get("panier"):
                d["statut"] = "panier"
            elif "remise en jeu" in str(d["reprise"]):
                d["panier"] = True
                d["statut"] = "à vérifier (cercle : non ; reprise : remise en jeu de l'autre équipe)"
            else:
                d["statut"] = "non"
        return d

    if FLEX:
        _verifier_tir = gv.vague_flex(_verifier_tir)
    with ThreadPoolExecutor(max_workers=PARALLELE if FLEX else 4) as _pool:
        _tirs_tries = sorted(tirs, key=lambda y: y["t"])
        _resultats = list(_pool.map(_verifier_tir, _tirs_tries))
    for x, d in zip(_tirs_tries, _resultats):
        reponses_seq.append(d)
        print(f"    tir {mmss(x['t'])} ({x['tireur']}) → cercle {d.get('panier')} | reprise : {d.get('reprise', '—')} "
              f"| {d.get('statut', '')} | {d.get('preuve', d.get('erreur', ''))}")
    if ECONOMIQUE:
        for x, d in zip(_tirs_tries, _resultats):
            r = str(x.get("reprise", ""))
            if d.get("panier") and ("lancers" in r or "sifflet" in r):
                x["decision"] = "panier ? (faute ensuite : annulé, lancer franc ou panier + faute)"
            elif d.get("panier"):
                x["decision"] = "panier (vérifié au cercle)"
            elif "remise en jeu" in r:
                x["decision"] = "panier ? (cercle : non ; reprise : remise en jeu de l'autre équipe)"
            else:
                x["decision"] = "raté"
        reponses_seq[:] = []
        if CIBLES:                         # essai ciblé : seuls les tirs revérifiés au cercle comptent
            _tt = list(_tirs_tries)
        for x in _tt:
            if str(x.get("decision", "")).startswith("panier"):
                reponses_seq.append({"panier": True, "instant": mmss(x["t"]), "marqueur": x["tireur2"],
                                     "equipe": "", "certitude": x["decision"], "nature": "balayage",
                                     "statut": "à vérifier" if "?" in x["decision"] else "panier"})
        # ── RÉSUMÉ ≤ 8 min : du tir le PLUS LOINTAIN au plus proche, les lancers francs en dernier (26/11 : on ne
        #    cherche pas à savoir si c'est un 3 points — la ligne n'est pas toujours visible) ──
        paniers = sorted([x for x in _tt if str(x.get("decision", "")).startswith("panier")],
                         key=lambda x: (x.get("lancer_franc", False), -x.get("distance", 0), x["t"]))
        resume, total = [], 0
        for x in paniers:
            if total + SEQ_AVANT_S + SEQ_APRES_S > RESUME_MAX_S:
                break
            resume.append(x); total += SEQ_AVANT_S + SEQ_APRES_S
        from collections import Counter as _C
        print(f"\n══ RÉSUMÉ (≤ {RESUME_MAX_S // 60} min) ══  {len(resume)}/{len(paniers)} panier(s), {total / 60:.1f} min | "
              f"lancers francs : {sum(1 for x in paniers if x.get('lancer_franc'))} | distances des autres : "
              f"{sorted([round(x['distance'], 1) for x in paniers if not x.get('lancer_franc')], reverse=True)}")
        print("  " + " · ".join(f"{mmss(x['t'])} {x['type']}" for x in sorted(resume, key=lambda y: y['t'])))
        rates = [x for x in _tt if not str(x.get("decision", "")).startswith("panier")]
        print(f"  « moments à voir » proposés à l'ajout : {len(paniers) - len(resume)} panier(s) hors résumé + {len(rates)} tir(s) raté(s)")
        # ── MONTAGE D'UN JOUEUR (exemple) ──
        num, coul = JOUEUR_TEST
        est = lambda v: bool(re.search(rf"\b{num}\b", str(v))) and coul in str(v).lower()
        ses_paniers = [x for x in paniers if est(x["tireur2"])]
        ses_passes = [x for x in paniers if est(x["passeur"])]
        ses_rebonds = [x for x in rates if est(x["rebondeur"])]
        vrais = [p for p, m in MARQUEURS.items() if m.split(" (")[0] == f"{num} {coul}"]
        trouves_j = [p for p in vrais if any(abs(sec(p) - x["t"]) <= 8 for x in ses_paniers)]
        print(f"\n══ MONTAGE DU N° {num} {coul.upper()} (exemple) ══")
        print(f"  ses paniers annoncés : {[mmss(x['t']) for x in ses_paniers]} | vrais : {vrais} → retrouvés {len(trouves_j)}/{len(vrais)}"
              f" | annoncés à tort : {len([x for x in ses_paniers if all(abs(sec(p) - x['t']) > 8 for p in vrais)])}")
        print(f"  ses passes décisives : {[mmss(x['t']) for x in ses_passes]} | ses rebonds : {[mmss(x['t']) for x in ses_rebonds]}")
        print("  (passes et rebonds : pas de vérité notée — à contrôler à l'œil)")
    fenetres = []                                                          # étape 3 (ancienne méthode) sautée
QUESTION_VIDEO = (
    "This is a 20-second clip of a youth basketball game filmed with a phone (mini-basket, low hoops). "
    "Is a BASKET SCORED (ball going through the hoop) in this clip? Decisive rule: AFTER A BASKET, play stops and "
    "the OTHER team restarts with a throw-in (inbound pass) from outside the court. If play simply CONTINUES after "
    "the shot (rebound, ball kept alive), there was NO basket. If the referee whistles and play restarts from the "
    "SIDE line, it is a foul or the ball went out, NOT a basket. Other clues: the ball drops through the net, the "
    "scorers celebrate. Answer ONLY with JSON: {\"panier\": true/false, "
    "\"instant_dans_extrait\": \"mm:ss from the START of this clip or ''\", \"equipe\": \"maillot couleur or ''\", "
    "\"marqueur\": \"numéro du tireur ou ''\", \"passeur\": \"numéro du passeur décisif ou ''\", "
    "\"reprise\": \"remise en jeu par l'autre equipe|le jeu continue|coup de sifflet, remise sur le cote|"
    "pas de tir|incertain\", \"certitude\": \"haute|moyenne|faible\", \"preuve\": \"one short sentence\"}")


def extrait_video(t0, duree=20):
    """Extrait de `duree` s à partir de t0, réduit (640 px, sans son) pour limiter le coût."""
    sortie = f"{SORTIE}/extrait_{int(t0)}.mp4"
    subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-ss", str(max(0, t0)), "-i", VIDEO, "-t", str(duree),
                    "-vf", "scale=640:-2", "-an", "-c:v", "libx264", "-crf", "30", "-preset", "veryfast", sortie], check=False)
    return open(sortie, "rb").read() if os.path.exists(sortie) else None


for nature, tc in fenetres:
    if SEQUENCE_EN_VIDEO:
        octets = extrait_video(tc - 10)
        if not octets:
            continue
        try:
            part = types.Part(inline_data=types.Blob(data=octets, mime_type="video/mp4"),
                              video_metadata=types.VideoMetadata(fps=FPS_VIDEO))
        except Exception:
            part = types.Part.from_bytes(data=octets, mime_type="video/mp4")
        try:
            r = client.models.generate_content(model="gemini-3.8-flash", contents=[part, QUESTION_VIDEO],
                                               config={"temperature": 0})
            txt = r.text or ""
            d = json.loads(txt[txt.index("{"):txt.rindex("}") + 1])
            rel = str(d.get("instant_dans_extrait") or "")
            if ":" in rel:
                d["instant"] = mmss(tc - 10 + sec(rel))
        except Exception as e:
            d = {"erreur": str(e)[:160]}
        d.update({"centre": mmss(tc), "nature": nature})
        reponses_seq.append(d)
        print(f"  {nature:6s} {mmss(tc)} → panier {d.get('panier')} {d.get('instant', '')} n° {d.get('marqueur', '')} "
              f"| reprise : {d.get('reprise', '?')} ({d.get('certitude', '')}) | {d.get('preuve', d.get('erreur', ''))}")
        continue
    contenu = [QUESTION_SEQ]
    for t in range(tc - 10, tc + 11, PAS_IMAGES_S):
        img = image_a(t)
        if img is None:
            continue
        img = cv2.resize(img, (960, int(960 * img.shape[0] / img.shape[1])))
        _, jpg = cv2.imencode(".jpg", img, [cv2.IMWRITE_JPEG_QUALITY, 80])
        contenu += [f"[{mmss(t)}]", types.Part.from_bytes(data=jpg.tobytes(), mime_type="image/jpeg")]
    try:
        r = client.models.generate_content(model="gemini-3.8-flash", contents=contenu, config={"temperature": 0})
        txt = r.text or ""
        d = json.loads(txt[txt.index("{"):txt.rindex("}") + 1])
    except Exception as e:
        d = {"erreur": str(e)[:120]}
    d.update({"centre": mmss(tc), "nature": nature})
    reponses_seq.append(d)
    print(f"  {nature:6s} {mmss(tc)} → panier {d.get('panier')} {d.get('instant', '')} n° {d.get('marqueur', '')} "
          f"| reprise : {d.get('reprise', '?')} ({d.get('certitude', '')}) | {d.get('preuve', d.get('erreur', ''))}")

# ── BILAN ──────────────────────────────────────────────────────────────────────────────────────
print("\n══ BILAN ══")
if BALAYAGE:
    # les séquences se chevauchent : un même panier vu deux fois (à moins de 8 s) ne compte qu'une fois
    trouves = []
    for d in reponses_seq:
        if d.get("panier") and d.get("instant") and ":" in str(d["instant"]):
            try:
                t_ = sec(d["instant"])
            except Exception:
                continue
            if all(abs(t_ - x[0]) > 8 for x in trouves):
                trouves.append((t_, d))
    print(f"  {len(trouves)} panier(s) annoncé(s) :")
    import re as _re
    n_num_ok = n_num = 0
    for t_, d in sorted(trouves, key=lambda x: x[0]):
        vrai = next((p for p in PANIERS_CONNUS if abs(sec(p) - t_) <= 8), None)
        incertain = any(abs(sec(p) - t_) <= 8 for p in INCERTAINS)
        annule = next((p for p in ANNULES_OU_RATES if abs(sec(p) - t_) <= 8), None)
        etat = ("✅ vrai panier " + vrai + " : " + MARQUEURS.get(vrai, "")) if vrai else \
               ("❔ panier incertain (annulé ?)" if incertain else
                f"❌ FAUSSE ALERTE (vérifié : {'annulé' if annule == '14:51' else 'lancer franc raté'})" if annule else
                "❌ FAUSSE ALERTE")
        if str(d.get("statut", "")).startswith("à vérifier"):
            etat += f"  [{d['statut']}]"
        if vrai:
            n_vrai = _re.findall(r"\d+", MARQUEURS.get(vrai, ""))[:1]
            n_lu = _re.findall(r"\d+", str(d.get("marqueur") or ""))[:1]
            if n_lu:
                n_num += 1
                n_num_ok += n_lu == n_vrai
                etat += "" if n_lu == n_vrai else f" (numéro lu {n_lu[0]} ≠ {n_vrai[0] if n_vrai else '?'})"
        print(f"    {mmss(t_)} {d.get('equipe', '')} n° {d.get('marqueur', '?')} [{d.get('certitude')}] → {etat}")
    if CIBLES:
        PANIERS_CONNUS = [p for p in PANIERS_CONNUS if p in CIBLES]
        print(f"  (essai ciblé : bilan sur les {len(PANIERS_CONNUS)} paniers ciblés seulement)")
    manques = [p for p in PANIERS_CONNUS if all(abs(sec(p) - t_) > 8 for t_, _ in trouves)]
    faux = sum(1 for t_, _ in trouves if all(abs(sec(p) - t_) > 8 for p in PANIERS_CONNUS + INCERTAINS))
    print(f"  PANIERS TROUVÉS : {len(PANIERS_CONNUS) - len(manques)}/{len(PANIERS_CONNUS)} | manqués : {manques or 'aucun'}"
          f" | fausses alertes : {faux} | marqueur lu juste : {n_num_ok}/{n_num} des paniers où un numéro est donné")
if PANIERS_CONNUS:
    vus = sum(1 for d in reponses_seq if d["nature"] == "connu" and d.get("panier"))
    print(f"  paniers connus retrouvés : {vus}/{len(PANIERS_CONNUS)}")
faux = sum(1 for d in reponses_seq if d["nature"] == "hasard" and d.get("panier"))
print(f"  séquences au hasard vues comme « panier » : {faux}/{sum(1 for d in reponses_seq if d['nature'] == 'hasard')}"
      f" (sans vérité : à contrôler à l'œil dans la vidéo)")
try:
    gv.gemini_cache_summary()
except Exception:
    pass
json.dump({"video": VIDEO, "resolution": [w, h], "fps": fps, "duree_min": duree / 60, "images": reponses_img,
           "sequences": reponses_seq}, open(f"{SORTIE}/bilan_basket.json", "w", encoding="utf-8"), ensure_ascii=False, indent=1)
with zipfile.ZipFile("/kaggle/working/test_basket.zip", "w") as z:
    for f in glob.glob(f"{SORTIE}/*"):
        z.write(f, os.path.basename(f))
print("✅ /kaggle/working/test_basket.zip (images annotées, réponses de Gemini, bilan)")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.9/79.9 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 43.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 102.7/102.7 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 580.6/580.6 kB 24.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.6/13.6 MB 98.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.0/35.0 MB 62.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 863.6/863.6 kB 32.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 25.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 84.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.2/126.2 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.3/4.3 MB 90.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.9/97.9 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.6/8

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.25.1 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
datasets 4.8.3 requires huggingface-hub<2.0,>=0.25.0, but you have huggingface-hub 2.5.0 which is incompatible.
diffusers 0.36.0 requires huggingface-hub<2.0,>=0.34.0, but you have huggingface-hub 2.5.0 which is incompatible.
gradio-client 1.14.0 requires huggingface-hub<2.0,>=0.19.3, but you have huggingface-hub 2.5.0 which is incompatible.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2026.2.0 which is incompatible.
gradio 5.50.0 requires huggingface-hub<2.0,>=0.33.5, but you have huggingface-hub 2.5.0 which is incompatible.


  [GEMINI] Verrou installé : modèle unique gemini-3.8-flash + cache pour tous les appels du projet
VIDÉO : /kaggle/input/datasets/michaelziant/basketjeune/basketJeune.mp4
  1332x748 | 30.0 images/s | 31.2 min | portion testée 0→1872 s

══ 1. DÉTECTION (RF-DETR, comme au football) ══
  (sautée : déjà faite)

══ 2. GEMINI, IMAGE PAR IMAGE ══

══ 3. GEMINI, SÉQUENCES DE 20 s : PANIER MARQUÉ ? ══
  étape 1 : repérage des tirs (images toutes les 2 s)
  [GEMINI CACHE] mode=on | 11782 réponse(s) en cache (19 fichier(s) lu(s)) | sortie : /kaggle/working/test_basket/gemini_cache_basket.jsonl
  panier situé sur une image en pleine taille : 72 tir(s) (1 sans panier visible, écartés)
  72 tir(s) repéré(s) : ['00:34 blanc', '01:04 ', '01:36 ', '01:42 30 vert', '02:06 ', '02:20 blanc', '02:24 ', '02:28 ', '03:02 14 blanc', '05:06 ', '05:36 blanc', '05:50 vert', '05:58 blanc', '06:22 31 vert', '06:42 vert', '06:48 31 vert', '07:28 vert', '07:54 14 blanc', '08:00 11 blanc', '08:08 blanc', '08:32 14 bl